# 04 · Validate — subunit-selectivity profile + stability vs native cytokine

**Standard slot:** *validate (in silico).* **For Project 13 this is the core analysis:** turn the
per-subunit `pae` into a **selectivity profile** (engages IL-2Rβ + γc but **not** IL-2Rα), quantify the
**selectivity margin**, and compare the de novo mimetics' **stability to the native cytokine** (the
Neo-2/15 selling point), with publication-style figures (D3 part 2).

Needs `results/all_ranked.csv` (notebook 03) and the design pool CSV(s) (notebook 02).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Build the selectivity profile

For each design we have `pae_to_alpha / beta / gamma`. A **βγ-biased** agonist (Neo-2/15-style) should
have **low** `pae` to β and γc (engaged signaling pair) and **high** `pae` to α (spared capture chain).
We compute, per design: `engaged_ok` (β and γc ≤ 10), `spared_ok` (α ≥ 14), and the **selectivity
margin** `= pae_to_alpha − max(pae_to_beta, pae_to_gamma)` (larger ⇒ cleaner βγ bias). Mock numbers are
SYNTHETIC; the thresholds are project-specific — justify yours.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
ENGAGE_MAX_PAE = 10.0   # β and γc must be <= this to count as ENGAGED (matches the binder cutoff)
SPARE_MIN_PAE  = 14.0   # α must be >= this to count as SPARED (project-specific; justify it)
MARGIN_MIN     = 4.0    # selectivity margin threshold (project-specific)

def selectivity_row(r):
    a, b, g = r.get("pae_to_alpha"), r.get("pae_to_beta"), r.get("pae_to_gamma")
    if pd.isna(a) or pd.isna(b) or pd.isna(g):
        return pd.Series(dict(engaged_ok=False, spared_ok=False, sel_margin=np.nan, selective=False))
    engaged_ok = (b <= ENGAGE_MAX_PAE) and (g <= ENGAGE_MAX_PAE)
    spared_ok  = (a >= SPARE_MIN_PAE)
    margin = a - max(b, g)
    selective = bool(engaged_ok and spared_ok and margin >= MARGIN_MIN)
    return pd.Series(dict(engaged_ok=engaged_ok, spared_ok=spared_ok,
                          sel_margin=round(float(margin), 2), selective=selective))

ranked = pd.concat([ranked, ranked.apply(selectivity_row, axis=1)], axis=1)

n = len(ranked)
n_eng_pass = int((ranked["layers_passed"] >= 3).sum())             # engaged the filter (nb 03)
n_selective = int(ranked["selective"].sum())                       # selective by profile
n_sel_and_pass = int((ranked["selective"] & (ranked["layers_passed"] >= 3)).sum())  # the real target
print(f"pool                              : {n}")
print(f"pass engaged filter (nb03)        : {n_eng_pass} ({100*n_eng_pass/max(n,1):.1f}%)")
print(f"selective by profile (βγ, not α)  : {n_selective} ({100*n_selective/max(n,1):.1f}%)")
print(f"SELECTIVE AGONIST CANDIDATES      : {n_sel_and_pass}  (pass filter AND selective)  [SYNTHETIC if mock]")
ranked.to_csv("results/all_ranked_selectivity.csv", index=False)
print("wrote results/all_ranked_selectivity.csv")

## 2 · Plot the selectivity profile

The clean separation we want: selective designs sit at **low `pae` to β/γc** and **high `pae` to α**.
We plot engaged-pae (worst of β/γc) vs α-pae; selective agonist candidates fall in the upper-left
(engaged & spared). The dashed lines are the thresholds.

In [ ]:
eng_pae = ranked[["pae_to_beta", "pae_to_gamma"]].max(axis=1)   # worst engaged subunit
a_pae = ranked["pae_to_alpha"]
sel = ranked["selective"].fillna(False).astype(bool)

fig, ax = plt.subplots(figsize=(5.4, 4.2))
ax.scatter(eng_pae[~sel], a_pae[~sel], s=16, alpha=0.5, label="not selective", color="#888888")
ax.scatter(eng_pae[sel],  a_pae[sel],  s=22, alpha=0.85, label="selective (βγ, not α)", color="#1f77b4")
ax.axvline(ENGAGE_MAX_PAE, ls="--", c="green", lw=1, label=f"engage ≤ {ENGAGE_MAX_PAE}")
ax.axhline(SPARE_MIN_PAE,  ls="--", c="red",   lw=1, label=f"spare α ≥ {SPARE_MIN_PAE}")
ax.set_xlabel("pae to engaged subunit (max of β, γc) — lower = engaged")
ax.set_ylabel("pae to IL-2Rα (CD25) — higher = spared")
ax.set_title("Subunit-selectivity profile (EXAMPLE_DATA if mock)")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.savefig("results/p13_selectivity.png", dpi=150); plt.show()
print("saved results/p13_selectivity.png")

## 3 · Selectivity margin distribution `[extension]`

The **selectivity margin** (`pae_to_α − max(pae_to_β, pae_to_γ)`) summarizes the profile in one number.
A right-shifted distribution means cleaner βγ bias. Compare paradigms if you ran both.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.4))
if "paradigm" in ranked.columns and ranked["paradigm"].notna().any():
    for p, gdf in ranked.groupby("paradigm"):
        ax.hist(gdf["sel_margin"].dropna(), bins=15, alpha=0.5, label=str(p))
    ax.legend(fontsize=8)
else:
    ax.hist(ranked["sel_margin"].dropna(), bins=15, alpha=0.7)
ax.axvline(MARGIN_MIN, ls="--", c="k", lw=1, label=f"margin ≥ {MARGIN_MIN}")
ax.set_xlabel("selectivity margin (Å):  pae_to_α − max(pae_to_β, pae_to_γ)")
ax.set_ylabel("designs"); ax.set_title("Selectivity margin (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p13_margin.png", dpi=150); plt.show()
print("median selectivity margin:", round(float(ranked["sel_margin"].median(skipna=True)), 2), "(SYNTHETIC if mock)")
print("saved results/p13_margin.png")

## 4 · Stability vs the native cytokine `[extension]`

The Neo-2/15 selling point: de novo mimetics can be **far more thermostable** than native IL-2 (no
disulfide dependence, idealized core). Here we **scaffold** the comparison with a clearly-labeled
EXAMPLE stability proxy — **on Colab replace it with a real proxy** (e.g., a short MD melting-RMSF, a
ΔΔG/FoldX-style score, or in the wet lab a **DSF Tm**). We anchor an EXAMPLE native-IL-2 reference value
purely so the bar chart renders; **it is not a measurement**.

In [ ]:
import cytokine_tools as ct

# EXAMPLE stability proxy (SYNTHETIC): a deterministic per-design "stability score" stand-in. On Colab,
# replace with a real proxy (MD RMSF / ΔΔG) or DSF Tm in the wet lab. NOT a measurement.
def example_stability_proxy(design_id):
    return round(45.0 + (ct._hashints("stab", design_id) % 350) / 10.0, 1)   # ~45-80 (arbitrary units)

NATIVE_IL2_PROXY = 55.0   # EXAMPLE_DATA anchor for native IL-2 on the SAME arbitrary scale (NOT a real Tm)

ranked["stability_proxy"] = ranked["design_id"].map(example_stability_proxy)
selective = ranked[ranked["selective"].fillna(False)]
med_design = float(ranked["stability_proxy"].median())
med_sel = float(selective["stability_proxy"].median()) if len(selective) else float("nan")

fig, ax = plt.subplots(figsize=(5.0, 3.4))
ax.bar(["native IL-2\n(EXAMPLE ref)", "all designs\n(median)", "selective\n(median)"],
       [NATIVE_IL2_PROXY, med_design, med_sel], color=["#bbbbbb", "#9ecae1", "#1f77b4"])
ax.set_ylabel("stability proxy (arbitrary, EXAMPLE_DATA)")
ax.set_title("Stability vs native cytokine (EXAMPLE_DATA — replace with MD/ΔΔG or DSF Tm)")
plt.tight_layout(); plt.savefig("results/p13_stability.png", dpi=150); plt.show()
print(f"median stability proxy — all designs: {med_design}, selective: {med_sel}, native ref: {NATIVE_IL2_PROXY}")
print("ALL stability numbers here are EXAMPLE_DATA/SYNTHETIC — replace with a real proxy/DSF and never report as measured.")
print("saved results/p13_stability.png")

## 5 · Select the top selective-agonist candidates

The D★ deliverable wants **subunit-selective agonist designs** carried forward. Rank the
filter-passing **and** selective designs by the composite score, tie-broken by selectivity margin, and
save the shortlist for the validation plan (notebook 05).

In [ ]:
cand = ranked[(ranked["layers_passed"] >= 3) & (ranked["selective"].fillna(False))].copy()
cand = cand.sort_values(["score", "sel_margin"], ascending=False).head(20)
cand.to_csv("results/top_candidates.csv", index=False)
print(f"wrote results/top_candidates.csv: {cand.shape} (filter-passing AND selective; top<=20)")
if "paradigm" in cand.columns:
    print("by paradigm:", cand.groupby("paradigm").size().to_dict())
cand.head(8)[["design_id", "paradigm", "score", "pae_to_alpha", "pae_to_beta", "pae_to_gamma", "sel_margin"]]

## D3 (part 2) checklist
- [ ] **Per-subunit selectivity profile** built (engages β+γc, spares α) with justified thresholds.
- [ ] Selectivity figure (`results/p13_selectivity.png`) + selectivity-margin distribution (`results/p13_margin.png`).
- [ ] **Stability-vs-native** comparison (`results/p13_stability.png`) — with a real proxy on Colab, EXAMPLE_DATA flagged here.
- [ ] Two honest numbers: engagement hit rate (nb03) **and** selective-agonist-candidate count (selective ∧ pass).
- [ ] `results/top_candidates.csv`: selective agonist candidates, ready for the validation plan.
- [ ] Stated plainly: selective in silico ≠ an agonist; binding ≠ signaling.

**Next:** `05_validation_plan.ipynb` — per-subunit SPR + the cell STAT-phosphorylation assay plan.